In [ ]:
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import yfinance as yf
import pandas_market_calendars as mcal
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly_express as px

In [2]:
dir_origen = '../DATOS/ORIGINALES'

dfs = [
    pd.read_csv(os.path.join(r, f))
    for r, _, files in os.walk(dir_origen)
    for f in files if f.lower().endswith('_marketstack.csv')
]

df = pd.concat(dfs, ignore_index=True)
df.to_csv("../DATOS/TRANSFORMADOS/dow_jones_consolidado.csv", index=False)

In [ ]:
df["date"] = pd.to_datetime(df["date"])
if df["date"].dt.tz is None:
    df["date"] = df["date"].dt.tz_localize("UTC")

df["date"] = df["date"].dt.tz_convert("America/New_York")

nyse = mcal.get_calendar("NYSE")
schedule = nyse.schedule(
    start_date=df["date"].min().date(), end_date=df["date"].max().date()
)
dias_habiles_nyse = schedule.index.date

es_dia_laborable = df["date"].dt.date.isin(dias_habiles_nyse)

hora_apertura = pd.to_datetime("09:30").time()
hora_cierre = pd.to_datetime("16:00").time()

es_horario_mercado = (df["date"].dt.time >= hora_apertura) & (
    df["date"].dt.time <= hora_cierre
)

df_horario_mercado = df[es_dia_laborable & es_horario_mercado].copy()

df_horario_mercado["date"] = df_horario_mercado["date"].dt.tz_localize(None)
df_horario_mercado

,open,high,low,last,close,volume,date,symbol,exchange
3,213.133,213.730,212.985,NaN,213.480,NaN,2018-08-16 16:00:00,AAPL,IEXG
4,213.273,213.300,213.133,NaN,213.133,NaN,2018-08-16 15:30:00,AAPL,IEXG
5,212.785,213.465,212.785,NaN,213.133,NaN,2018-08-16 15:00:00,AAPL,IEXG
6,212.850,213.198,212.785,NaN,212.785,NaN,2018-08-16 14:30:00,AAPL,IEXG
7,211.780,213.198,211.780,NaN,212.785,NaN,2018-08-16 14:00:00,AAPL,IEXG
...,...,...,...,...,...,...,...,...,...
2744413,234.500,235.600,234.170,235.300,234.090,291094.0,2021-06-29 13:07:00,V,IEXG
2744414,234.500,235.600,234.170,235.235,234.090,290571.0,2021-06-29 13:05:00,V,IEXG
2744415,234.500,235.600,234.170,235.270,234.090,290254.0,2021-06-29 13:04:00,V,IEXG
2744416,234.500,235.600,234.170,235.255,234.090,289854.0,2021-06-29 13:03:00,V,IEXG


In [ ]:
df_horario_mercado["fecha_dia"] = df_horario_mercado["date"].dt.date
dias_reales = df_horario_mercado.groupby("symbol")["fecha_dia"].unique()

dias_faltantes = [
    {"symbol": activo, "fecha_faltante": dia.strftime("%Y-%m-%d")}
    for activo, fechas in dias_reales.items()
    for dia in set(pd.bdate_range(min(fechas), max(fechas)).date) - set(fechas)
]

df_dias_faltantes = pd.DataFrame(dias_faltantes).sort_values(
    ["symbol", "fecha_faltante"]
)
df_dias_faltantes

if not df_dias_faltantes.empty:
    tickers = df_dias_faltantes["symbol"].unique().tolist()
    f_min = df_dias_faltantes["fecha_faltante"].min()
    f_max = (
        pd.to_datetime(df_dias_faltantes["fecha_faltante"].max())
        + pd.Timedelta(days=1)
    ).strftime("%Y-%m-%d")

    df_yf_dias = (
        yf.download(
            tickers, start=f_min, end=f_max, interval="1d", group_by="ticker"
        )
        .stack(level=0, future_stack=True)
        .reset_index()
    )
    df_yf_dias = df_yf_dias.rename(
        columns={
            "Date": "date",
            "Ticker": "symbol",
            "Open": "open",
            "High": "high",
            "Low": "low",
            "Close": "close",
            "Volume": "volume",
        }
    )
    df_yf_dias["date"] = pd.to_datetime(df_yf_dias["date"])

    df_dias_rec = pd.merge(
        df_dias_faltantes,
        df_yf_dias,
        left_on=["symbol", "fecha_faltante"],
        right_on=["symbol", df_yf_dias["date"].dt.strftime("%Y-%m-%d")],
    ).drop(columns=["fecha_faltante"])
else:
    df_dias_rec = pd.DataFrame()

cols = ["symbol", "date", "open", "high", "low", "close", "volume"]
df_consolidado_completo = (
    pd.concat([df_horario_mercado[cols], df_dias_rec[cols]], ignore_index=True)
    .drop_duplicates(subset=["symbol", "date"])
    .sort_values(["symbol", "date"])
    .reset_index(drop=True)
)

[*********************100%***********************]  26 of 26 completed


In [5]:
df_pivot = df_consolidado_completo.pivot(index='date', columns='symbol').swaplevel(axis=1).sort_index(axis=1)
columnas_validas = ["open", "high", "low", "close", "volume"]
df_pivot = df_pivot.loc[:, (slice(None), columnas_validas)]

In [6]:
agg_dict = {}
for col in df_pivot.columns:
    ticker, metric = col
    
    if metric == 'open':
        agg_dict[col] = 'first' 
    elif metric == 'high':
        agg_dict[col] = 'max'  
    elif metric == 'low':
        agg_dict[col] = 'min'  
    elif metric == 'close':
        agg_dict[col] = 'last' 
    elif metric == 'volume':
        agg_dict[col] = 'sum'  

df_daily = df_pivot.groupby(df_pivot.index.normalize()).agg(agg_dict)

df_daily = df_daily.dropna(how='all')

In [7]:
tickers = df['symbol'].unique().tolist()

datos_yf = yf.download(tickers, start="2018-06-06", end="2021-06-30", auto_adjust=False, actions=True)

splits_data = datos_yf['Stock Splits']

tickers_with_splits = splits_data.loc[:, (splits_data > 0).any(axis=0)]

actual_splits = tickers_with_splits[(tickers_with_splits > 0).any(axis=1)]

print(actual_splits.dropna(how='all'))

[*********************100%***********************]  26 of 26 completed


Ticker      AAPL    MRK
Date                   
2020-08-31   4.0  0.000
2021-06-03   0.0  1.048


In [ ]:
prices_to_adjust = ['open', 'high', 'low', 'close']

for split_date, row in actual_splits.iterrows():
    split_date_str = split_date.strftime('%Y-%m-%d')
    
    for ticker, ratio in row.items():
        if ratio > 0.0:
            print(f"Applying {ratio}-for-1 split adjustment for {ticker} prior to and including {split_date_str}")
            
            pre_split_mask = df_daily.index.normalize() <= split_date_str
            
            for col in prices_to_adjust:
                df_daily.loc[pre_split_mask, (ticker, col)] /= ratio
            
            df_daily.loc[pre_split_mask, (ticker, 'volume')] *= ratio

Applying 4.0-for-1 split adjustment for AAPL prior to and including 2020-08-31
Applying 1.048-for-1 split adjustment for MRK prior to and including 2021-06-03

All historical splits applied successfully.


In [ ]:
window = 5
threshold = 0.15
for tk in df_daily.columns.get_level_values(0).unique():
    if (tk, 'close') in df_daily.columns:
        close_series = df_daily[tk]['close']
        
        rolling_med = close_series.rolling(window=window, center=True).median()
        
        deviation = np.abs((close_series - rolling_med) / rolling_med)
        
        df_daily.loc[deviation > threshold, (tk, 'close')] = np.nan
        df_daily[(tk, 'close')] = df_daily[(tk, 'close')].interpolate(method='linear')

In [ ]:
tickers = df_daily.columns.get_level_values(0).unique().tolist()
fecha_inicio = df_daily.index.min().strftime('%Y-%m-%d')
fecha_fin = '2025-06-30'

df_yf = yf.download(
    tickers, 
    start=fecha_inicio, 
    end=fecha_fin, 
    auto_adjust=False
)

df_yf = df_yf[['Open', 'High', 'Low', 'Close', 'Volume']]

df_yf = df_yf.swaplevel(axis=1)

df_yf.columns = df_yf.columns.set_levels(df_yf.columns.levels[1].str.lower(), level=1)

df_yf.columns.names = df_daily.columns.names

if df_yf.index.tz is not None:
    df_yf.index = df_yf.index.tz_localize(None)

df_daily_completo = df_daily.combine_first(df_yf)

df_daily_completo = df_daily_completo.sort_index(axis=1)
orden_metricas = ["open", "high", "low", "close", "volume"]
df_daily_completo = df_daily_completo.loc[:, (slice(None), orden_metricas)]

df_daily_completo = df_daily_completo.dropna(how='all')

df_daily_completo.head()

[*********************100%***********************]  26 of 26 completed


symbol,AAPL,AMGN,AXP,BA,CAT,CRM,CSCO,CVX,DIS,DOW,...,KO,MCD,MMM,MRK,MSFT,NKE,PG,TRV,UNH,V
,open,open,open,open,open,open,open,open,open,open,...,volume,volume,volume,volume,volume,volume,volume,volume,volume,volume
date,,,,,,,,,,,,,,,,,,,,,
2018-06-06,48.40750,182.009995,100.000000,362.459991,153.649994,134.330002,43.759998,123.120003,100.059998,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2018-06-07,48.53875,183.470001,101.440002,372.000000,157.110001,135.550003,44.240002,124.540001,101.910004,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2018-06-08,47.74500,183.889999,101.290001,368.140015,156.190002,131.699997,43.520000,127.120003,102.089996,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2018-06-11,47.83625,184.139999,101.180000,371.000000,155.110000,133.880005,43.619999,126.769997,103.930000,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2018-06-12,47.84625,184.270004,101.500000,371.239990,156.690000,134.960007,43.680000,127.639999,104.730003,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
df_daily_completo = df_daily_completo.ffill()

tickers = df_daily_completo.columns.get_level_values(0).unique().tolist()
fecha_inicio = df_daily_completo.index.min().strftime('%Y-%m-%d')
fecha_fin = '2025-06-30'

datos_yf_check = yf.download(
    tickers, 
    start=fecha_inicio, 
    end=fecha_fin, 
    auto_adjust=False, 
    actions=True
)

splits_check = datos_yf_check['Stock Splits']

tickers_con_splits = splits_check.loc[:, (splits_check > 0).any(axis=0)]
splits_detectados = tickers_con_splits[(tickers_con_splits > 0).any(axis=1)]

print("Splits:")
print(splits_detectados.dropna(how='all'))

[*********************100%***********************]  26 of 26 completed

Todos los splits detectados en la ventana temporal:
Ticker      AAPL    IBM    MMM    MRK
Date                                 
2020-08-31   4.0  0.000  0.000  0.000
2021-06-03   0.0  0.000  0.000  1.048
2021-11-04   0.0  1.046  0.000  0.000
2024-04-01   0.0  0.000  1.196  0.000


In [ ]:
splits_pendientes = {
    'IBM': ('2021-11-04', 1.046),
    'MMM': ('2024-04-01', 1.196)
}

columnas_precio = ['open', 'high', 'low', 'close']

for ticker, (fecha, ratio) in splits_pendientes.items():
    mascara_pre_split = df_daily_completo.index <= fecha
    
    for col in columnas_precio:
        df_daily_completo.loc[mascara_pre_split, (ticker, col)] /= ratio
        
    df_daily_completo.loc[mascara_pre_split, (ticker, 'volume')] *= ratio

Splits de IBM y MMM ajustados correctamente en df_daily_completo.


In [ ]:
tickers = df_daily_completo.columns.get_level_values(0).unique()

for ticker in tickers:
    close_col = df_daily_completo[(ticker, 'close')]
    high_col = df_daily_completo[(ticker, 'high')]
    low_col = df_daily_completo[(ticker, 'low')]
    
    mascara_anomalia = (close_col > high_col) | (close_col < low_col)
    
    df_daily_completo.loc[mascara_anomalia, (ticker, 'close')] = np.nan
    
    if (ticker, 'last') in df_daily_completo.columns:
        df_daily_completo[(ticker, 'close')] = df_daily_completo[(ticker, 'close')].fillna(
            df_daily_completo[(ticker, 'last')].ffill()
        )
    else:
        df_daily_completo[(ticker, 'close')] = df_daily_completo[(ticker, 'close')].fillna(
            df_daily_completo[(ticker, 'close')].shift(1)
        )
        
    df_daily_completo[(ticker, 'close')] = df_daily_completo[(ticker, 'close')].bfill()

    df_daily_completo[(ticker, 'high')] = df_daily_completo[ticker][['open', 'high', 'low', 'close']].max(axis=1)
    df_daily_completo[(ticker, 'low')] = df_daily_completo[ticker][['open', 'high', 'low', 'close']].min(axis=1)

Imputación con 'last' y consistencia estructural aplicada con éxito.


In [ ]:
inconsistencias = {}
tickers = df_daily_completo.columns.get_level_values(0).unique()

for ticker in tickers:
    df_t = df_daily_completo[ticker]
    problemas = []
    
    nulos = df_t.isna().sum().sum()
    if nulos > 0:
        problemas.append(f"{nulos} NAs detectados.")
        
    negativos = (df_t < 0).sum().sum()
    if negativos > 0:
        problemas.append(f"{negativos} negativos detectados.")
        
    high_menor_low = (df_t['high'] < df_t['low']).sum()
    if high_menor_low > 0:
        problemas.append(f"{high_menor_low} días donde High < Low.")
        
    high_invalido = ((df_t['high'] < df_t['open']) | (df_t['high'] < df_t['close'])).sum()
    if high_invalido > 0:
        problemas.append(f"{high_invalido} días donde High es menor que Open o Close.")
        
    low_invalido = ((df_t['low'] > df_t['open']) | (df_t['low'] > df_t['close'])).sum()
    if low_invalido > 0:
        problemas.append(f"{low_invalido} días donde Low es mayor que Open o Close.")
        
    primer_dia_cotizacion = df_daily_completo[(ticker, 'close')].first_valid_index()
    
    if primer_dia_cotizacion is not None:
        serie_valida = df_daily_completo.loc[primer_dia_cotizacion:, (ticker, 'close')]
        df_daily_completo.loc[primer_dia_cotizacion:, (ticker, 'close')] = serie_valida.bfill(limit=3)
    if problemas:
        inconsistencias[ticker] = problemas

if inconsistencias:
    print("Se han encontrado las siguientes inconsistencias:")
    for ticker, lista_prob in inconsistencias.items():
        print(f"\n{ticker}:")
        for p in lista_prob:
            print(f"  - {p}")
else:
    print("No hay inconsistencias.")

⚠️ Se encontraron las siguientes inconsistencias:

DOW:
  - 197 valores nulos (NaNs) detectados.


In [ ]:
# Tras el filtro anterior, quitamos los datos de DWO que se habían interpolado para ajustar la serie a la real (empieza a cotizar más tarde que el resto)

ipo_dow = '2020-05-29'

mascara_pre_ipo = df_daily_completo.index < ipo_dow

df_daily_completo.loc[mascara_pre_ipo, ('DOW', slice(None))] = np.nan

Historial previo a la IPO de DOW limpiado con éxito.


In [ ]:
raw_tickers = df_daily_completo.columns.get_level_values(0).unique()
tickers_to_plot = [tk for tk in raw_tickers if isinstance(tk, str) and tk.isupper() and 1 <= len(tk) <= 5]

fig = make_subplots(
    rows=len(tickers_to_plot), 
    cols=1, 
    shared_xaxes=True, 
    subplot_titles=tickers_to_plot
)

for i, tk in enumerate(tickers_to_plot, start=1):
    ticker_close = df_daily_completo[tk]['close'].dropna()
    
    fig.add_trace(
        go.Scatter(
            x=ticker_close.index, 
            y=ticker_close.values, 
            mode='lines', 
            name=tk,
            line=dict(width=1.5)
        ),
        row=i, 
        col=1
    )

fig.update_layout(
    height=300 * len(tickers_to_plot),
    title_text="Histórico de valores Close",
    template="plotly_white",
    showlegend=False
)

fig.show()

In [17]:
df_daily_completo.to_parquet("../DATOS/TRANSFORMADOS/df_diario.parquet")